# Memory Consolidation Experiments - Kaggle GPU

This notebook runs heavy compute experiments on Kaggle's free T4/P100 GPUs.

**Resource Strategy:**
- Orchestration: macpro51 (local)
- Embeddings: completeu-server Ollama
- Heavy training: This notebook (Kaggle)

**Free GPU Limits:**
- 30 GPU-hours per week
- 9-hour session limit
- T4 or P100 GPU

In [ ]:
# Install dependencies
!pip install -q sentence-transformers numpy pandas matplotlib seaborn

In [ ]:
# Verify GPU
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Experiment Code

The following cells contain the core experiment logic. This runs independently from the local orchestrator.

In [ ]:
# Core experiment classes - standalone version for Kaggle

from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional
from datetime import datetime
import numpy as np
import time
import json
from sentence_transformers import SentenceTransformer

# Use GPU for embeddings
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")

@dataclass
class Memory:
    id: str
    content: str
    timestamp: datetime
    memory_type: str
    embedding: Optional[np.ndarray] = None
    significance: float = 0.5
    access_count: int = 0

class EmbeddingEngine:
    """GPU-accelerated embedding engine for Kaggle."""
    def __init__(self, model_name='all-MiniLM-L6-v2'):
        print(f"Loading embedding model on {DEVICE}...")
        self.model = SentenceTransformer(model_name, device=DEVICE)
        print("Model loaded!")
    
    def encode(self, texts: List[str]) -> np.ndarray:
        return self.model.encode(texts, convert_to_numpy=True, show_progress_bar=True)

# Initialize embedding engine
embedding_engine = EmbeddingEngine()

In [ ]:
# Task generation for experiments
import random
from enum import Enum

class TaskType(Enum):
    BUG_FIX = "bug_fix"
    FEATURE_ADD = "feature_add"
    REFACTOR = "refactor"
    DEBUG = "debug"

COMPONENTS = ["UserService", "OrderProcessor", "PaymentHandler", "AuthManager",
              "DataLoader", "CacheManager", "MessageQueue", "TaskScheduler"]

def generate_task(session_id: int, task_id: int) -> Dict:
    task_type = random.choice(list(TaskType))
    component = random.choice(COMPONENTS)
    return {
        'id': f'task_{session_id}_{task_id}',
        'session_id': session_id,
        'type': task_type.value,
        'component': component,
        'title': f'{task_type.value.replace("_", " ").title()} in {component}',
        'description': f'Perform {task_type.value} on {component} module',
        'difficulty': random.choice([1, 2, 3])
    }

def generate_dataset(num_sessions=10, tasks_per_session=5, seed=42):
    random.seed(seed)
    sessions = []
    for s in range(num_sessions):
        tasks = [generate_task(s, t) for t in range(tasks_per_session)]
        sessions.append({'id': s, 'tasks': tasks})
    return sessions

# Generate dataset
dataset = generate_dataset(num_sessions=10, tasks_per_session=5)
print(f"Generated {len(dataset)} sessions with {sum(len(s['tasks']) for s in dataset)} total tasks")

In [ ]:
# Agent implementations

class BaseAgent:
    def __init__(self, name: str):
        self.name = name
        self.memories: List[Memory] = []
    
    def store(self, content: str, **kwargs) -> Memory:
        mem_id = f"{self.name}_{len(self.memories)}"
        embedding = embedding_engine.encode([content])[0]
        memory = Memory(
            id=mem_id,
            content=content,
            timestamp=datetime.now(),
            memory_type=kwargs.get('memory_type', 'flat'),
            embedding=embedding,
            significance=kwargs.get('significance', 0.5)
        )
        self.memories.append(memory)
        return memory
    
    def retrieve(self, query: str, k: int = 5) -> List[Memory]:
        if not self.memories:
            return []
        query_emb = embedding_engine.encode([query])[0]
        similarities = []
        for mem in self.memories:
            sim = np.dot(query_emb, mem.embedding) / (
                np.linalg.norm(query_emb) * np.linalg.norm(mem.embedding) + 1e-8
            )
            similarities.append((mem, sim))
        similarities.sort(key=lambda x: x[1], reverse=True)
        return [m for m, _ in similarities[:k]]
    
    def consolidate(self) -> Dict:
        return {'consolidated': False, 'agent': self.name}

class FlatMemoryAgent(BaseAgent):
    """Baseline: Simple flat memory."""
    pass

class ConsolidationAgent(BaseAgent):
    """Experimental: Multi-tier with consolidation."""
    def __init__(self, name: str):
        super().__init__(name)
        self.semantic_memory: List[Dict] = []
        self.procedural_memory: List[Dict] = []
    
    def consolidate(self) -> Dict:
        # Pattern extraction from memories
        patterns_found = 0
        content_words = {}
        for mem in self.memories:
            for word in mem.content.lower().split():
                content_words[word] = content_words.get(word, 0) + 1
        
        # Extract frequent patterns
        for word, count in content_words.items():
            if count >= 3 and len(word) > 4:
                self.semantic_memory.append({
                    'concept': word,
                    'frequency': count,
                    'confidence': min(1.0, count / 10)
                })
                patterns_found += 1
        
        return {
            'consolidated': True,
            'patterns_extracted': patterns_found,
            'semantic_concepts': len(self.semantic_memory)
        }

In [ ]:
# Run Experiment 1: Consolidation vs Flat Memory

def run_experiment(agent, dataset, consolidate_interval=2):
    """Run experiment and collect metrics."""
    results = {
        'agent_type': agent.name,
        'sessions': [],
        'task_results': []
    }
    
    for session in dataset:
        session_results = []
        
        for task in session['tasks']:
            # Store task as memory
            content = f"Task: {task['title']}. {task['description']}"
            agent.store(content, significance=task['difficulty'] / 3)
            
            # Retrieve relevant context
            retrieved = agent.retrieve(task['title'], k=5)
            
            # Simulate success based on retrieval quality
            base_prob = {1: 0.9, 2: 0.7, 3: 0.5}[task['difficulty']]
            context_boost = 0.1 * len(retrieved)
            success_prob = min(1.0, base_prob + context_boost)
            success = random.random() < success_prob
            
            session_results.append({
                'task_id': task['id'],
                'success': success,
                'retrieved_count': len(retrieved)
            })
        
        results['sessions'].append({
            'session_id': session['id'],
            'success_rate': sum(r['success'] for r in session_results) / len(session_results)
        })
        results['task_results'].extend(session_results)
        
        # Consolidate periodically
        if session['id'] % consolidate_interval == 0:
            cons_result = agent.consolidate()
            results['sessions'][-1]['consolidation'] = cons_result
    
    # Final metrics
    results['overall_success_rate'] = sum(r['success'] for r in results['task_results']) / len(results['task_results'])
    results['total_memories'] = len(agent.memories)
    
    return results

print("Running experiments...")
print()

# Run with both agents
random.seed(42)
flat_agent = FlatMemoryAgent('flat_memory')
flat_results = run_experiment(flat_agent, dataset, consolidate_interval=100)  # No consolidation
print(f"Flat Memory: {flat_results['overall_success_rate']:.1%} success")

random.seed(42)
cons_agent = ConsolidationAgent('consolidation')
cons_results = run_experiment(cons_agent, dataset, consolidate_interval=2)
print(f"Consolidation: {cons_results['overall_success_rate']:.1%} success")
print(f"  Semantic concepts learned: {len(cons_agent.semantic_memory)}")

In [ ]:
# Visualize results
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Success rate comparison
ax1 = axes[0]
agents = ['Flat Memory', 'Consolidation']
success_rates = [flat_results['overall_success_rate'], cons_results['overall_success_rate']]
ax1.bar(agents, success_rates, color=['steelblue', 'coral'])
ax1.set_ylabel('Success Rate')
ax1.set_title('Overall Task Success Rate')
ax1.set_ylim(0, 1)
for i, v in enumerate(success_rates):
    ax1.text(i, v + 0.02, f'{v:.1%}', ha='center')

# Learning curve
ax2 = axes[1]
flat_sessions = [s['success_rate'] for s in flat_results['sessions']]
cons_sessions = [s['success_rate'] for s in cons_results['sessions']]
ax2.plot(flat_sessions, 'o-', label='Flat Memory')
ax2.plot(cons_sessions, 's-', label='Consolidation')
ax2.set_xlabel('Session')
ax2.set_ylabel('Success Rate')
ax2.set_title('Learning Curve Across Sessions')
ax2.legend()
ax2.set_ylim(0, 1)

plt.tight_layout()
plt.savefig('experiment_results.png', dpi=150)
plt.show()

In [ ]:
# Save results
all_results = {
    'experiment': 'consolidation_vs_flat',
    'timestamp': datetime.now().isoformat(),
    'flat_memory': flat_results,
    'consolidation': cons_results
}

with open('experiment_results.json', 'w') as f:
    json.dump(all_results, f, indent=2, default=str)

print("Results saved to experiment_results.json")
print(f"\nConsolidation improvement: {(cons_results['overall_success_rate'] - flat_results['overall_success_rate'])*100:.1f}% absolute")

## Experiment 2: Consolidation Frequency Ablation

Test different consolidation intervals: Never, every 1 session, every 2, every 5

In [ ]:
# Run frequency ablation
intervals = [100, 1, 2, 5]  # 100 = effectively never
interval_names = ['Never', 'Every 1', 'Every 2', 'Every 5']
ablation_results = []

for interval, name in zip(intervals, interval_names):
    random.seed(42)
    agent = ConsolidationAgent(f'cons_{name}')
    result = run_experiment(agent, dataset, consolidate_interval=interval)
    ablation_results.append({
        'interval': name,
        'success_rate': result['overall_success_rate'],
        'semantic_concepts': len(agent.semantic_memory)
    })
    print(f"{name}: {result['overall_success_rate']:.1%} success, {len(agent.semantic_memory)} concepts")

# Plot ablation
plt.figure(figsize=(8, 4))
plt.bar([r['interval'] for r in ablation_results], [r['success_rate'] for r in ablation_results])
plt.ylabel('Success Rate')
plt.title('Consolidation Frequency Ablation')
plt.ylim(0, 1)
plt.savefig('ablation_results.png', dpi=150)
plt.show()